# GKX (2019) — Pipeline Replication
**Authors:** Francesco Pio De Girolamo · Brice Namy — IEOR E4733 · Spring 2026  

This notebook drives the `main.py` CLI pipelines end-to-end. The two core training variants are:
- `paper`    — strict GKX (1957–2016, no transaction costs)
- `improved` — extended sample (1957–2024) + impact-aware transaction costs

The repository also exposes extension and forward-scenario variants (`extended_2024`, `extended_ciz_2026`, `post2016_ciz`, and the `future2026_*` family). The general CLI form for any of them is:
```bash
python main.py --mode train --variant <variant> --models <model list>
```
Each variant writes to its own `outputs/<variant>/` directory and uses its own cached feature matrix at `data/cache/feature_matrix_<variant>.parquet`, so the pipelines do **not** overwrite each other. The Streamlit dashboard (`src/dashboard/app.py`) reads those committed `outputs/<variant>/` artifacts directly — no retraining required to view results.

**Workflow:** run Section 0 once on a fresh runtime; then Section 1 after every restart; then Sections 2–5 to drive each variant; Section 6 launches the dashboard.

## Section 0 — First-time setup
Run these cells **once** on a brand-new Colab runtime.  After a runtime restart, skip straight to Section 1.

### 0.1 Mount Google Drive & extract project

In [ ]:
# Mount Drive and extract the v3 project zip (FIRST TIME ONLY).
import os, zipfile
from google.colab import drive

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/Algo Trading Project/empirical_asset_pricing_ml.zip'
PROJECT  = '/content/empirical_asset_pricing_ml'

if not os.path.exists(PROJECT):
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(
            f"Place the zip at: {ZIP_PATH}\n"
            "or change ZIP_PATH above."
        )
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('/content')
    print(f'✅ Extracted project to {PROJECT}')
else:
    print(f'ℹ️  Project already extracted at {PROJECT}')


### 0.2 Install dependencies

In [ ]:
%cd /content/empirical_asset_pricing_ml
!pip install -r requirements.txt --quiet
!pip install wrds --no-deps --quiet


# Section 1 — Post-restart setup
Run these cells **every time** the runtime restarts. They are lightweight: set the Python path, restore the cache from Drive, apply runtime patches, and verify GPU availability. No extraction, no pip.

### 1.1 Python path & working directory

In [ ]:
import sys, os, pathlib
PROJECT = '/content/empirical_asset_pricing_ml'
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)
print('Working directory:', os.getcwd())


### 1.2 GPU and RAM check

In [ ]:
import torch, psutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
print(f'RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')


### 1.3 Restore cache + results from Drive
The canonical Drive backup lives at `cache_backup/` and `result_backup/`. This restores both, preserving the variant-specific subdirs `outputs/paper/`, `outputs/improved/`, and the variant-specific feature parquets.

In [ ]:
import shutil, pathlib

DRIVE_BASE   = pathlib.Path('/content/drive/MyDrive/Algo Trading Project')
CACHE_BACKUP = DRIVE_BASE / 'cache_backup'
CACHE_LOCAL  = pathlib.Path('data/cache')
if CACHE_BACKUP.exists():
    shutil.copytree(str(CACHE_BACKUP), str(CACHE_LOCAL), dirs_exist_ok=True)
    items = sorted(p.name for p in CACHE_LOCAL.iterdir())
    print(f'✅ Cache restored ({len(items)} items):', items)
else:
    print('ℹ️  No cache backup found on Drive — will build from scratch.')

OUTPUTS_BACKUP = DRIVE_BASE / 'outputs_backup'
OUTPUTS_LOCAL  = pathlib.Path('outputs')
if OUTPUTS_BACKUP.exists():
    shutil.copytree(str(OUTPUTS_BACKUP), str(OUTPUTS_LOCAL), dirs_exist_ok=True)
    print(f'✅ Outputs restored from {OUTPUTS_BACKUP}')
else:
    print('ℹ️  No outputs backup found on Drive — starting fresh.')


### 1.4 Patch `all_models.py` — enable GPU for neural networks

In [ ]:
import torch
MODELS_FILE = 'src/models/all_models.py'
with open(MODELS_FILE, 'r') as f:
    src = f.read()
old = 'device: str = "cpu"'
new = 'device: str = "cuda" if torch.cuda.is_available() else "cpu"'
if old in src:
    src = src.replace(old, new, 1)
    with open(MODELS_FILE, 'w') as f:
        f.write(src)
    print(f'✅ GPU patch applied. Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
elif new in src:
    print('✅ GPU patch already applied.')
else:
    print('⚠️  Target string not found — investigate.')


### 1.5 Backup helper

In [ ]:
import shutil, os
from datetime import datetime

DRIVE_BASE_STR = str(DRIVE_BASE)

def backup_to_drive(label: str = ''):
    """Backup local cache and outputs to Drive.

    - cache   → cache_backup/    (canonical, always overwritten)
    - outputs → outputs_backup/  (canonical, always overwritten, includes both variants)
    """
    pairs = [
        ('data/cache', os.path.join(DRIVE_BASE_STR, 'cache_backup')),
        ('outputs',    os.path.join(DRIVE_BASE_STR, 'outputs_backup')),
    ]
    for src, dst in pairs:
        if os.path.exists(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'  ✅ {src} → {dst}')
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    if label:
        print(f'[{stamp}] Backup complete ({label})')


# Section 2 — Choose variant
Set `VARIANT` once for this kernel, then re-run the notebook (or just Sections 3–5) per variant. Supported values:

| Variant                          | Mode        | Notes |
|----------------------------------|-------------|-------|
| `paper`                          | `train`     | Strict GKX 1957–2016, TC=0. |
| `improved`                       | `train`     | Extended to 2024 + impact-aware TC. Recommended source for `predict`. |
| `extended_2024`                  | `train`     | Real-only post-paper extension on legacy `crsp.msf` (to 2024-12-31). |
| `extended_ciz_2026`              | `train`     | CIZ/v2-aware extension to 2026-03-31. |
| `post2016_ciz`                   | `predict`   | Scoring-only 2017–2026; reuses pickles from `improved` (or `paper`). |
| `future2026_{base,trending,mean_reversion,rotating_leaders,choppy,crisis,factor_rotation}` | `predict` | Forward scenarios; reuse `improved` pickles. |

Canonical commands (any variant):
```bash
python main.py --mode train    --variant <variant> --models <model list>
python main.py --mode evaluate --variant <variant>
python main.py --mode regimes  --variant <variant>
python main.py --mode importance --variant <variant>
python main.py --mode predict  --variant <target> --source-model-variant improved
```


In [ ]:
# ⚠️  CHANGE THIS to switch pipelines.
VARIANT = 'paper'      # 'paper' or 'improved'
WRDS_USERNAME = 'brice77'   # only used in Section 3 (data fetch)
print('Variant:', VARIANT)


In [ ]:
# Show the resolved settings
from src.config import get_variant_config
cfg = get_variant_config(VARIANT)
print('Resolved settings for', VARIANT)
for k, v in cfg.items():
    print(f'  {k:28s} {v}')


# Section 3 — Data pipeline
Run these **once per variant**.  Step 1 is identical between variants if their date ranges overlap; the WRDS cache is shared by date range. Steps 2–4 are variant-specific.

If you already ran the data pipeline on Drive and Section 1.3 restored it, you can skip this section entirely.

### 3.1 Fetch raw data (WRDS + GWZ macro)

In [ ]:
!python main.py --mode data-only --data-step fetch --variant $VARIANT --wrds-username $WRDS_USERNAME


In [ ]:
# (Same thing using the Python interpolation, in case the shell-arg version misbehaves.)
import subprocess
subprocess.run([
    'python', 'main.py', '--mode', 'data-only', '--data-step', 'fetch',
    '--variant', VARIANT, '--wrds-username', WRDS_USERNAME,
], check=True)


### 3.2 Merge CRSP + Compustat

In [ ]:
import subprocess
subprocess.run(['python', 'main.py', '--mode', 'data-only', '--data-step', 'merge', '--variant', VARIANT], check=True)


### 3.3 Build characteristics

In [ ]:
import subprocess
subprocess.run(['python', 'main.py', '--mode', 'data-only', '--data-step', 'chars', '--variant', VARIANT], check=True)


### 3.4 Build feature matrix (Kronecker product + macro interactions + SIC dummies)

In [ ]:
import subprocess
subprocess.run(['python', 'main.py', '--mode', 'data-only', '--data-step', 'features', '--variant', VARIANT], check=True)


### 3.5 Backup data cache to Drive

In [ ]:
backup_to_drive(f'data_{VARIANT}')

# Section 4 — Training
Each stage below calls `python main.py --mode train --variant {VARIANT} --models ...` for a model subset. ⚠️ **Restart the runtime between stages** to free RAM. After restart, re-run Section 1 + Section 2 (the cell where `VARIANT` is set).

All training results land in `outputs/<VARIANT>/models/<model>.pkl`. Already-trained models are skipped automatically — pass `--force-retrain` to override.

## Stage 4a: Linear models (CPU, fast)

In [ ]:
import subprocess
for m in ['OLS-3', 'ENet+H', 'PCR', 'PLS', 'GLM+H']:
    subprocess.run(['python', 'main.py', '--mode', 'train', '--variant', VARIANT, '--models', m], check=False)
backup_to_drive(f'linear_{VARIANT}')


## Stage 4b: Tree models
*Restart runtime first, then re-run Section 1 + Section 2.*

In [ ]:
import subprocess
for m in ['GBRT+H', 'RF']:
    subprocess.run(['python', 'main.py', '--mode', 'train', '--variant', VARIANT, '--models', m], check=False)
backup_to_drive(f'trees_{VARIANT}')


## Stage 4c: Neural networks
*Restart runtime first, re-run Section 1 + 2. GPU is auto-detected.*

In [ ]:
import subprocess
for m in ['NN1', 'NN2', 'NN3', 'NN4', 'NN5']:
    subprocess.run(['python', 'main.py', '--mode', 'train', '--variant', VARIANT, '--models', m], check=False)
    backup_to_drive(f'{m}_{VARIANT}')


# Section 5 — Evaluate + Variable Importance
Compute OOS R², H-L Sharpe (net & gross), Diebold-Mariano stats and p-values, Sharpe\* (Campbell-Thompson), MaxDD, Skew, Kurtosis, Alpha and t(α). All written to `outputs/<VARIANT>/`.

Variable importance is a separate step because it requires fitting each model on the train+val window once (the backtest engine refits each year and discards models).

### 5.1 Evaluate

In [ ]:
import subprocess
subprocess.run(['python', 'main.py', '--mode', 'evaluate', '--variant', VARIANT], check=True)
backup_to_drive(f'eval_{VARIANT}')


### 5.2 Inspect comprehensive metrics

In [ ]:
import pandas as pd
out = pd.read_csv(f'outputs/{VARIANT}/comprehensive.csv', index_col=0)
print('=== Comprehensive metrics ===')
out


### 5.3 Inspect Diebold-Mariano matrix

In [ ]:
import pandas as pd
dm   = pd.read_csv(f'outputs/{VARIANT}/dm_table.csv',   index_col=0)
pval = pd.read_csv(f'outputs/{VARIANT}/dm_pvalues.csv', index_col=0)
print('=== DM stats ===')
print(dm.round(2))
print()
print('=== DM p-values (two-sided) ===')
print(pval.round(3))


### 5.4 Regime-conditional evaluation
Slices each model's H-L return series by NBER recessions, VIX terciles, and decade. Reads the saved portfolio bundle from `outputs/<VARIANT>/portfolio_returns.pkl` and writes `regimes.csv`. **Run after `--mode evaluate`.**

In [ ]:
import subprocess
subprocess.run(['python', 'main.py', '--mode', 'regimes', '--variant', VARIANT], check=True)
backup_to_drive(f'regimes_{VARIANT}')


In [ ]:
import pandas as pd
reg = pd.read_csv(f'outputs/{VARIANT}/regimes.csv')
print('=== H-L Sharpe (net) by NBER regime ===')
print(reg[reg['regime_kind']=='nber']
        .pivot(index='model', columns='regime', values='sharpe_net')
        .round(3))
print()
print('=== H-L Sharpe (net) by VIX tercile ===')
print(reg[reg['regime_kind']=='vix']
        .pivot(index='model', columns='regime', values='sharpe_net')
        .round(3))


### 5.5 Variable importance
This step **fits each model on train+val once** then perturbs each base characteristic on a test slice and measures the drop in OOS R². Results are aggregated from the 920 Kronecker features back to the 94 base characteristics.

Default method is `zero` (set feature to its cross-sectional median = 0). Use `--importance-method permutation` for the alternative.

In [ ]:
# Pick which models to compute importance for (subset is fine — this is slow for NNs).
IMPORTANCE_MODELS = ['OLS-3', 'ENet+H', 'PCR', 'PLS', 'GBRT+H']

import subprocess
subprocess.run([
    'python', 'main.py',
    '--mode', 'importance',
    '--variant', VARIANT,
    '--importance-method', 'zero',
    '--models', *IMPORTANCE_MODELS,
], check=True)
backup_to_drive(f'importance_{VARIANT}')


### 5.6 Inspect variable importance

In [ ]:
import pandas as pd
imp = pd.read_csv(f'outputs/{VARIANT}/var_importance.csv', index_col=0)
print('Top 20 base characteristics by max importance across models:')
imp.assign(max_imp=imp.abs().max(axis=1)).sort_values('max_imp', ascending=False).head(20).drop(columns='max_imp')


# Section 5bis — Predict mode for forward / extension variants
The `post2016_ciz` and `future2026_*` variants are **scoring-only**: they do not retrain models. Instead, `--mode predict` slices the per-model prediction pickles from a *source* training variant (typically `improved`, whose pickles span 1987–2024) into the target variant's test window, so you obtain OOS results without any new training.

Canonical pattern:
```bash
python main.py --mode predict --variant <target_variant> \
    --source-model-variant improved
```

After `predict`, you still run `--mode evaluate` (and optionally `--mode regimes`) on the target variant to build its `comprehensive.csv`, `dm_table.csv`, `portfolio_returns.pkl`, and `regimes.csv`.


### 5bis.1 Run predict for the `post2016_ciz` window (2017–2026, CIZ-aware)

In [ ]:
import subprocess
TARGET = 'post2016_ciz'
SOURCE = 'improved'  # also valid: 'paper'
subprocess.run([
    'python', 'main.py',
    '--mode', 'predict',
    '--variant', TARGET,
    '--source-model-variant', SOURCE,
], check=True)
subprocess.run(['python', 'main.py', '--mode', 'evaluate', '--variant', TARGET], check=True)
subprocess.run(['python', 'main.py', '--mode', 'regimes',  '--variant', TARGET], check=True)
backup_to_drive(f'predict_{TARGET}')


### 5bis.2 Run predict for each `future2026_*` scenario
Each scenario lives under its own `outputs/future2026_<scenario>/` and reuses the same trained-model pickles from `improved`.

In [ ]:
import subprocess
SOURCE = 'improved'
FUTURE_SCENARIOS = [
    'future2026_base',
    'future2026_trending',
    'future2026_mean_reversion',
    'future2026_rotating_leaders',
    'future2026_choppy',
    'future2026_crisis',
    'future2026_factor_rotation',
]
for v in FUTURE_SCENARIOS:
    print(f'\n=== {v} ===')
    subprocess.run(['python', 'main.py', '--mode', 'predict',  '--variant', v, '--source-model-variant', SOURCE], check=True)
    subprocess.run(['python', 'main.py', '--mode', 'evaluate', '--variant', v], check=True)
    subprocess.run(['python', 'main.py', '--mode', 'regimes',  '--variant', v], check=True)
    backup_to_drive(f'predict_{v}')


### 5bis.3 Quick summary of `comprehensive.csv` across scoring variants

In [ ]:
import pandas as pd, pathlib
ROWS = []
for v in ['post2016_ciz'] + FUTURE_SCENARIOS:
    f = pathlib.Path(f'outputs/{v}/comprehensive.csv')
    if f.exists():
        df = pd.read_csv(f, index_col=0)
        df.insert(0, 'variant', v)
        ROWS.append(df)
if ROWS:
    summary = pd.concat(ROWS)
    print(summary.head(40))
else:
    print('No comprehensive.csv found yet — run cells 5bis.1 / 5bis.2 first.')


# Section 6 — Dashboard
Launch the Streamlit dashboard. Use the **variant selector** in the sidebar to flip between paper and improved results.

In [ ]:
# In Colab, expose Streamlit via ngrok (or Colab's port forwarding).
# Streamlit recipe is the same locally — just `streamlit run src/dashboard/app.py`.
!pip install pyngrok --quiet
!pip install streamlit --quiet


In [ ]:
# Launch (Colab helper). Adapt for your environment if needed.
import subprocess, time
from pyngrok import ngrok

# Start streamlit in the background
proc = subprocess.Popen(
    ['streamlit', 'run', 'src/dashboard/app.py',
     '--server.port', '8501', '--server.headless', 'true'],
)
time.sleep(5)
public_url = ngrok.connect(8501).public_url
print(f'Streamlit dashboard: {public_url}')


# Section 7 — Convenience: run both core variants end-to-end
Use this only if you have plenty of compute time. It chains the full pipeline (`--mode data-only` → `--mode train --variant <variant>` → `--mode evaluate` → `--mode importance`) back-to-back for `paper` and `improved`. For Colab, prefer the staged Sections 3–5 with runtime restarts.

For the post-2016 / forward-scenario variants, swap in `--mode predict --source-model-variant improved` instead of `--mode train` to reuse the already-trained models.

In [ ]:
import subprocess

for v in ['paper', 'improved']:
    print(f'\n========== Running variant: {v} ==========')
    subprocess.run(['python', 'main.py', '--mode', 'data-only', '--variant', v,
                    '--wrds-username', WRDS_USERNAME], check=True)
    subprocess.run(['python', 'main.py', '--mode', 'train', '--variant', v,
                    '--models', 'OLS-3', 'ENet+H', 'PCR', 'PLS'], check=False)
    subprocess.run(['python', 'main.py', '--mode', 'evaluate', '--variant', v], check=True)
    subprocess.run(['python', 'main.py', '--mode', 'importance', '--variant', v,
                    '--models', 'OLS-3', 'ENet+H', 'PCR', 'PLS'], check=False)
    backup_to_drive(f'all_{v}')


# Section 8 — Final backup of code + outputs

In [ ]:
import os, shutil
backup_dir = str(DRIVE_BASE / 'outputs_backup')
os.makedirs(backup_dir, exist_ok=True)
for folder in ['outputs', 'logs']:
    src = f'/content/empirical_asset_pricing_ml/{folder}'
    if os.path.exists(src):
        dst = os.path.join(backup_dir, folder)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
print('✅ Outputs & logs backed up to:', backup_dir)


In [ ]:
import subprocess
from datetime import datetime
stamp    = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f'empirical_asset_pricing_ml_code_{stamp}.zip'
zip_path = str(DRIVE_BASE / zip_name)
subprocess.run([
    'zip', '-r', zip_path, 'empirical_asset_pricing_ml',
    '--exclude', '*/data/cache/*',
    '--exclude', '*/data/gwz_extracted/*',
    '--exclude', '*/__pycache__/*',
    '--exclude', '*/.git/*',
    '--exclude', '*/outputs/*',
    '--exclude', '*/logs/*',
], cwd='/content', check=True)
print('✅ Code snapshot saved at:', zip_path)
